# FINANCE 361 — Topic 13: Anomalies and Factor Models
## Interactive Python Notebook

**University of Auckland | Dr. Zicheng (Leo) Xiao**

---

This notebook covers:
1. Downloading real **Fama-French factor data** from Ken French's website
2. Comparing the **CAPM, FF3, and FF5** as benchmark models
3. Building and testing a **momentum anomaly**
4. Replicating the **microcap bias** demonstrated by Hou, Xue & Zhang (2018)
5. Visualising the **replication rate** by methodology


## Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn statsmodels pandas-datareader --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
import pandas_datareader.data as web
from IPython.display import display

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('All packages loaded.')

---
## Part 1 — Download Real Fama-French Factor Data

Ken French maintains a public data library at Dartmouth. We download:
- **FF3**: MKTRF, SMB, HML (+ RF)
- **FF5**: adds CMA, RMW
- **Momentum (UMD)**: needed for Carhart 4-factor model


In [ ]:
START, END = '2000-01', '2023-12'

try:
    # FF5 factors
    ff5_raw = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench',
                              start=START, end=END)[0] / 100
    ff5_raw.index = pd.to_datetime(ff5_raw.index.to_timestamp())

    # Momentum factor
    mom_raw = web.DataReader('F-F_Momentum_Factor', 'famafrench',
                              start=START, end=END)[0] / 100
    mom_raw.index = pd.to_datetime(mom_raw.index.to_timestamp())
    mom_raw.columns = ['UMD']

    factors = ff5_raw.join(mom_raw, how='inner')
    factors.index.name = 'date'
    print(f'Downloaded {len(factors)} months of FF5 + momentum data.')
    display(factors.head(5))

except Exception as e:
    print(f'Download failed ({e}). Generating simulated factors...')
    n = 288  # 24 years
    idx = pd.date_range('2000-01', periods=n, freq='MS')
    # Simulated factors with realistic correlations
    mkt  = np.random.normal(0.006, 0.045, n)
    factors = pd.DataFrame({
        'Mkt-RF': mkt,
        'SMB':    np.random.normal(0.002, 0.025, n),
        'HML':    np.random.normal(0.003, 0.027, n),
        'RMW':    np.random.normal(0.003, 0.020, n),
        'CMA':    np.random.normal(0.002, 0.018, n),
        'RF':     np.random.uniform(0.0000, 0.0004, n),
        'UMD':    np.random.normal(0.007, 0.038, n),
    }, index=idx)
    factors.index.name = 'date'
    display(factors.head(5))

---
## Part 2 — Factor Cumulative Returns

Plot the **cumulative return** of each factor over the full sample. This shows the long-run evidence for each premium.


In [ ]:
factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']
factor_labels = {
    'Mkt-RF': 'Market (MKTRF)',
    'SMB':    'Size (SMB)',
    'HML':    'Value (HML)',
    'RMW':    'Profitability (RMW)',
    'CMA':    'Investment (CMA)',
    'UMD':    'Momentum (UMD)',
}
colors = ['#2563eb', '#16a34a', '#d97706', '#7c3aed', '#0891b2', '#dc2626']

cumret = (1 + factors[factor_cols]).cumprod() - 1

fig, ax = plt.subplots(figsize=(12, 5))
for col, color in zip(factor_cols, colors):
    ax.plot(cumret.index, cumret[col] * 100, label=factor_labels[col], color=color, linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Cumulative Factor Returns (2000–2023)', fontweight='bold', fontsize=13)
ax.set_ylabel('Cumulative Return (%)')
ax.legend(loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('topic13_cumulative_factors.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAnnualised mean returns by factor:')
ann = ((1 + factors[factor_cols].mean()) ** 12 - 1) * 100
display(ann.to_frame('Annualised Mean Return (%)').round(2))

---
## Part 3 — Compare CAPM vs FF3 vs FF5 on a Test Portfolio

We construct a simple **momentum-based** test portfolio (similar to UMD) and test it against CAPM, FF3, and FF5.

The key question: **how much of the momentum portfolio's return can each model explain?**


In [ ]:
# Use UMD (momentum) as the test portfolio
# (In practice this would be YOUR anomaly strategy's return series)
y = factors['UMD'].copy()

models_spec = {
    'CAPM':  ['Mkt-RF'],
    'FF3':   ['Mkt-RF', 'SMB', 'HML'],
    'FFC4':  ['Mkt-RF', 'SMB', 'HML', 'UMD'],  # note: UMD on RHS
    'FF5':   ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA'],
}

results = {}
for name, factors_list in models_spec.items():
    rhs = factors_list if name != 'FFC4' else [f for f in factors_list if f != 'UMD']
    X = sm.add_constant(factors[rhs])
    m = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 6})
    results[name] = {
        'alpha_bps': m.params['const'] * 10000,       # basis points per month
        't_alpha':   m.tvalues['const'],
        'r2':        m.rsquared,
        'n':         int(m.nobs),
    }

res_df = pd.DataFrame(results).T
res_df.columns = ['Alpha (bps/month)', 't-statistic', 'R²', 'N']
res_df['Significant?'] = res_df['t-statistic'].abs().gt(1.96).map({True: '✓ Yes', False: '✗ No'})

print('=== Momentum Portfolio Alpha Under Different Benchmark Models ===')
print('(Using UMD as the test portfolio)')
print()
display(res_df.round(3))

In [ ]:
# Visualise alphas and R² across models
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_names = list(results.keys())
alphas  = [results[m]['alpha_bps'] for m in model_names]
t_stats = [results[m]['t_alpha']   for m in model_names]
r2s     = [results[m]['r2']        for m in model_names]

bar_colors = ['#dc2626' if abs(t) > 1.96 else '#94a3b8' for t in t_stats]

bars = axes[0].bar(model_names, alphas, color=bar_colors, edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Alpha by Benchmark Model\n(red = significant |t|>1.96)', fontweight='bold')
axes[0].set_ylabel('Alpha (basis points / month)')
for bar, val, t in zip(bars, alphas, t_stats):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{val:.0f}bp\n(t={t:.1f})', ha='center', va='bottom', fontsize=8.5)

axes[1].bar(model_names, [r*100 for r in r2s], color='#2563eb', alpha=0.75)
axes[1].set_title('R² by Benchmark Model\n(how much return variation is explained)', fontweight='bold')
axes[1].set_ylabel('R² (%)')
for i, (m, r) in enumerate(zip(model_names, r2s)):
    axes[1].text(i, r*100+0.5, f'{r*100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('topic13_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4 — The Microcap Bias (Hou, Xue & Zhang 2018)

HXZ (2018) showed that most anomalies disappear when you:
1. Use **NYSE breakpoints** (not all-stock breakpoints)
2. Use **value-weighted returns** (not equal-weighted)

The mechanism: microcap stocks have inflated equal-weighted returns. Let's demonstrate this with simulated data.


In [ ]:
# Simulate a universe of 500 stocks with different market caps
np.random.seed(123)
n_stocks, n_months = 500, 120

# Market caps: log-normally distributed (mimics real stock universe)
log_mktcap = np.random.normal(5, 2, n_stocks)   # log(market cap in $M)
mktcap = np.exp(log_mktcap)                      # skewed: many small, few large

# Is each stock listed on NYSE (top 30% by size)?
is_nyse = mktcap >= np.percentile(mktcap, 70)
is_microcap = mktcap < np.percentile(mktcap, 20)  # bottom 20% = microcaps

# Generate a sorting variable x with genuine signal + noise
signal_x = np.random.normal(0, 1, n_stocks)

# ── Simulate returns ───────────────────────────────────────────────────────
# True alpha differs by size:
# Large stocks: modest but real return to signal
# Microcaps: huge apparent return (bid-ask bounce + illiquidity)
true_alpha_large    = 0.002  # 0.2% per month
true_alpha_microcap = 0.015  # 1.5% per month (spurious!)

alpha_per_stock = np.where(is_microcap, true_alpha_microcap, true_alpha_large)
noise = np.random.normal(0, 0.08, (n_months, n_stocks))

# Returns: alpha * signal_x (scaled) + noise
returns = alpha_per_stock * signal_x[None, :] + noise  # shape (n_months, n_stocks)

# ── Assign quintiles using ALL stocks vs NYSE only ─────────────────────────
def quintile_5m1(signal, returns, weights=None, nyse_mask=None):
    """Compute 5M1 returns using given breakpoint method and weighting."""
    monthly_returns = []
    for t in range(n_months):
        # Breakpoints based on NYSE or all stocks
        if nyse_mask is not None:
            breaks = np.percentile(signal[nyse_mask], [20, 40, 60, 80])
        else:
            breaks = np.percentile(signal, [20, 40, 60, 80])
        q = np.digitize(signal, breaks) + 1  # quintiles 1-5
        q = np.clip(q, 1, 5)
        r = returns[t]
        if weights is None:  # equal-weighted
            q5_ret = r[q == 5].mean()
            q1_ret = r[q == 1].mean()
        else:                # value-weighted
            w = weights
            q5_ret = (r[q==5] * w[q==5]).sum() / w[q==5].sum()
            q1_ret = (r[q==1] * w[q==1]).sum() / w[q==1].sum()
        monthly_returns.append(q5_ret - q1_ret)
    return np.array(monthly_returns)

# Four combinations
all_ew   = quintile_5m1(signal_x, returns)
all_vw   = quintile_5m1(signal_x, returns, weights=mktcap)
nyse_ew  = quintile_5m1(signal_x, returns, nyse_mask=is_nyse)
nyse_vw  = quintile_5m1(signal_x, returns, weights=mktcap, nyse_mask=is_nyse)

print('Mean 5M1 returns by methodology:')
method_results = pd.DataFrame({
    'Methodology': ['All-EW', 'All-VW', 'NYSE-EW', 'NYSE-VW'],
    'Mean 5M1 (%/month)': [all_ew.mean()*100, all_vw.mean()*100,
                            nyse_ew.mean()*100, nyse_vw.mean()*100],
    't-statistic': [
        sm.OLS(all_ew,  sm.add_constant(np.ones(n_months))).fit().tvalues[0],
        sm.OLS(all_vw,  sm.add_constant(np.ones(n_months))).fit().tvalues[0],
        sm.OLS(nyse_ew, sm.add_constant(np.ones(n_months))).fit().tvalues[0],
        sm.OLS(nyse_vw, sm.add_constant(np.ones(n_months))).fit().tvalues[0],
    ]
})
method_results['Significant?'] = method_results['t-statistic'].abs().gt(1.96).map({True:'✓ Yes', False:'✗ No'})
display(method_results.round(3))

In [ ]:
# Visualise the microcap bias
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: Market cap distribution ─────────────────────────────────────────
axes[0].hist(np.log10(mktcap), bins=40, color='#94a3b8', edgecolor='white')
axes[0].axvline(np.log10(np.percentile(mktcap, 20)), color='#dc2626', linestyle='--',
                linewidth=2, label='Microcap threshold (P20)')
axes[0].axvline(np.log10(np.percentile(mktcap, 70)), color='#2563eb', linestyle='--',
                linewidth=2, label='NYSE threshold (P70)')
axes[0].set_xlabel('log₁₀(Market Cap)')
axes[0].set_ylabel('Number of stocks')
axes[0].set_title('Stock Universe: Market Cap Distribution', fontweight='bold')
axes[0].legend(fontsize=8.5)

n_micro = is_microcap.sum()
n_large = (~is_microcap & is_nyse).sum()
axes[0].text(0.97, 0.97,
             f'Microcaps: {n_micro} ({n_micro/n_stocks*100:.0f}% of stocks)\n'
             f'Large (NYSE): {n_large} ({n_large/n_stocks*100:.0f}% of stocks)',
             transform=axes[0].transAxes, ha='right', va='top',
             fontsize=8.5, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# ── Right: 5M1 means by methodology ───────────────────────────────────────
methods = ['All-EW', 'All-VW', 'NYSE-EW', 'NYSE-VW']
means_  = [all_ew.mean()*100, all_vw.mean()*100, nyse_ew.mean()*100, nyse_vw.mean()*100]
t_vals  = method_results['t-statistic'].values
bcolors = ['#2563eb' if abs(t) > 1.96 else '#94a3b8' for t in t_vals]

bars = axes[1].bar(methods, means_, color=bcolors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Mean 5M1 Return by Methodology\n(blue = significant |t|>1.96)', fontweight='bold')
axes[1].set_ylabel('Mean Monthly Return (%)')
for bar, val, t in zip(bars, means_, t_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, max(bar.get_height(), 0) + 0.01,
                 f'{val:.2f}%\n(t={t:.1f})', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('topic13_microcap_bias.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key insight: All-EW inflates returns due to microcap stocks.')
print(f'Microcap true alpha: {true_alpha_microcap*100:.1f}%/month (spurious)',
      f'  vs  Large stock true alpha: {true_alpha_large*100:.1f}%/month (genuine)')

---
## Part 5 — Recreate the HXZ (2018) Replication Rate Chart

Figure 2 from Hou, Xue & Zhang (2018) shows the fraction of 452 anomalies that replicate under different methodologies.


In [ ]:
# Data from HXZ (2018) Figure 2, Panel A (extended sample Jan 1967–Dec 2016)
hxz_data = pd.DataFrame({
    'Methodology':   ['FM-OLS', 'FM-WLS', 'All-EW', 'All-VW', 'NYSE-EW', 'NYSE-VW'],
    'Single_test':   [33.6,     58.2,     56.4,     58.6,     40.7,      35.0],
    'Multiple_test': [13.3,     48.5,     46.7,     48.0,     22.1,      17.9],
})
hxz_data = hxz_data.set_index('Methodology')

fig, ax = plt.subplots(figsize=(10, 5))

y_pos  = np.arange(len(hxz_data))
height = 0.35

b1 = ax.barh(y_pos + height/2, hxz_data['Single_test'],  height, color='#2563eb', alpha=0.85, label='Single test (|t|≥1.96)')
b2 = ax.barh(y_pos - height/2, hxz_data['Multiple_test'], height, color='#94a3b8', alpha=0.85, label='Multiple test (|t|≥2.78)')

for bar in b1:
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontsize=8.5, color='#2563eb')
for bar in b2:
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontsize=8.5, color='#64748b')

ax.set_yticks(y_pos)
ax.set_yticklabels(hxz_data.index, fontsize=10)
ax.set_xlabel('Replication Rate (% of 452 anomalies)', fontsize=10)
ax.set_title('Anomaly Replication Rates by Methodology\n'
             'Source: Hou, Xue & Zhang (2018), Figure 2 Panel A', fontweight='bold', fontsize=11)
ax.axvline(35, color='#dc2626', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlim(0, 75)
ax.legend(loc='lower right', fontsize=9)

# Annotation
ax.annotate('NYSE-VW: only 35%\nreplicate (single test)',
            xy=(35, 0), xytext=(50, 0.8),
            arrowprops=dict(arrowstyle='->', color='#dc2626'),
            fontsize=8.5, color='#dc2626')

plt.tight_layout()
plt.savefig('topic13_replication_rates.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey finding: With the STRICTEST methodology (NYSE-VW),')
print(f'only {hxz_data.loc["NYSE-VW","Single_test"]:.0f}% of anomalies clear the single-test bar.')
print(f'Only {hxz_data.loc["NYSE-VW","Multiple_test"]:.1f}% clear the multiple-test bar.')

---
## Part 6 — Factor Correlation Matrix

How correlated are the Fama-French factors with each other? High correlations suggest redundancy; low correlations suggest each factor captures something distinct.


In [ ]:
corr = factors[factor_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    ax=ax, linewidths=0.5,
    xticklabels=[factor_labels[c] for c in factor_cols],
    yticklabels=[factor_labels[c] for c in factor_cols],
)
ax.set_title('Pairwise Correlations: Fama-French Factors', fontweight='bold')
plt.tight_layout()
plt.savefig('topic13_factor_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlation interpretation:')
print('- Near 0: factors capture independent variation')
print('- Near ±1: factors are redundant')

---
## Part 7 — Build Your Own Sample Filter (CRSP-Style)

Practice applying the standard data filters from HXZ (2018). Modify the parameters to see how the sample size changes.


In [ ]:
# Simulate a CRSP-like universe for one month
np.random.seed(7)
n_raw = 8000  # all listed securities in a given month

raw = pd.DataFrame({
    'permno':   range(n_raw),
    'shrcd':    np.random.choice([10, 11, 12, 14, 18, 48, 73], size=n_raw,
                                  p=[0.35, 0.15, 0.10, 0.10, 0.10, 0.10, 0.10]),
    'exchcd':   np.random.choice([1, 2, 3, 4], size=n_raw, p=[0.25, 0.15, 0.45, 0.15]),
    'ret':      np.random.normal(0.01, 0.08, n_raw),
    'mktcap':   np.exp(np.random.normal(4, 2.5, n_raw)),
    'sic':      np.random.choice(range(1000, 9000), size=n_raw),
})
# Introduce ~10% missing returns
raw.loc[np.random.choice(n_raw, size=800, replace=False), 'ret'] = np.nan

print(f'Raw universe: {len(raw):,} securities')

# ── Apply filters ──────────────────────────────────────────────────────────
# Parameters you can adjust:
DROP_FINANCE  = True   # drop SIC 6000-6999
DROP_UTILITIES = False # drop SIC 4900-4999
DROP_MICROCAPS = True  # drop bottom 20% of NYSE mktcap

sample = raw.copy()
print(f'After step 0 (raw):                   {len(sample):>6,}')

sample = sample[sample['shrcd'].isin([10, 11])]
print(f'After step 1 (common shares only):    {len(sample):>6,}')

sample = sample[sample['exchcd'].isin([1, 2, 3])]
print(f'After step 2 (NYSE/AMEX/NASDAQ):      {len(sample):>6,}')

sample = sample.dropna(subset=['ret', 'mktcap'])
print(f'After step 3 (non-missing ret & mkt): {len(sample):>6,}')

if DROP_FINANCE:
    sample = sample[~sample['sic'].between(6000, 6999)]
    print(f'After step 4 (drop finance SIC 6xxx): {len(sample):>6,}')

if DROP_UTILITIES:
    sample = sample[~sample['sic'].between(4900, 4999)]
    print(f'After step 4b (drop utilities):       {len(sample):>6,}')

if DROP_MICROCAPS:
    nyse_p20 = sample[sample['exchcd'] == 1]['mktcap'].quantile(0.20)
    sample = sample[sample['mktcap'] >= nyse_p20]
    print(f'After step 5 (drop microcaps NYSE-P20):{len(sample):>6,}')

pct = len(sample) / n_raw * 100
print(f'\nFinal sample: {len(sample):,} ({pct:.1f}% of raw universe)')
print('Consistent with HXZ (2018): "Less than half of stocks survive typical filters."')

---
## Summary: Key Takeaways

| Topic | Key Finding |
|-------|-------------|
| Factor models | CAPM → FF3 → FF5/HXZ4 — each generation explains more cross-sectional variation |
| Anomalies | Hundreds documented; most started as anomalies before being co-opted as factors |
| Microcap bias | Equal-weighting + all-stock breakpoints inflate anomaly returns dramatically |
| Replication | With NYSE-VW methodology, only ~35% of 452 anomalies replicate (HXZ 2018) |
| Best practice | Use NYSE breakpoints, value-weighted returns, multiple-testing correction |

---
*FINANCE 361 | University of Auckland | Dr. Zicheng (Leo) Xiao*
